<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/04_Feature_Engineering_Deposits_Forecast_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# БЛОК 4: FEATURE ENGINEERING — ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ И ПОДГОТОВКА БАЗОВОЙ МОДЕЛИ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Создание базовых признаков (лаги DEPOS)
df['DEPOS_log'] = np.log(df['DEPOS'])
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# Подготовка X и y
X_base = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X_base.index, 'DEPOS']

# Разделение на train/test
train_size = len(X_base) - 12
X_train_base, X_test_base = X_base.iloc[:train_size], X_base.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# Масштабирование для базовой модели
scaler_base = StandardScaler()
X_train_scaled_base = scaler_base.fit_transform(X_train_base)
X_test_scaled_base = scaler_base.transform(X_test_base)

# Базовая модель Ridge
ridge_base = Ridge(alpha=1.0)
ridge_base.fit(X_train_scaled_base, y_train)
y_pred_base = ridge_base.predict(X_test_scaled_base)

r2_base = r2_score(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)

print("="*60)
print("БАЗОВАЯ МОДЕЛЬ (RIDGE, ALPHA=1.0)")
print("="*60)
print(f"R² = {r2_base:.4f}")
print(f"RMSE = {rmse_base:.2f} млрд руб.")
print(f"MAE = {mae_base:.2f} млрд руб.")

# ============================================================
# 3. ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ
# ============================================================

print("\n" + "="*60)
print("ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ")
print("="*60)

# Копируем данные для новых признаков
df_feat = df.copy()

# 3.1. Сезонные фиктивные переменные
print("\n🔍 Добавление сезонных фиктивных переменных...")
df_feat['Month'] = df_feat.index.month
month_dummies = pd.get_dummies(df_feat['Month'], prefix='month', drop_first=True)
df_feat = pd.concat([df_feat, month_dummies], axis=1)
print(f"   Добавлено 11 сезонных переменных")

# 3.2. Фиктивная переменная post_2022
print("\n🔍 Добавление фиктивной переменной post_2022...")
df_feat['post_2022'] = (df_feat.index >= '2023-01-01').astype(int)
print(f"   post_2022 = 1 для {df_feat['post_2022'].sum()} записей (с 2023 года)")

# 3.3. Фиктивная переменная covid
print("\n🔍 Добавление фиктивной переменной covid...")
df_feat['covid'] = ((df_feat.index >= '2020-03-01') & (df_feat.index <= '2022-01-01')).astype(int)
print(f"   covid = 1 для {df_feat['covid'].sum()} записей")

# 3.4. Фиктивная переменная regime_cred1 (по DEPOS)
print("\n🔍 Добавление фиктивной переменной regime_cred1...")
threshold_depos = 32000
df_feat['regime_cred1'] = (df_feat['DEPOS'] > threshold_depos).astype(int)
print(f"   regime_cred1 = 1 для {df_feat['regime_cred1'].sum()} записей (DEPOS > {threshold_depos})")

# 3.5. Фиктивная переменная anomaly_wage (с адаптивным порогом)
print("\n🔍 Добавление фиктивной переменной anomaly_wage с адаптивным порогом...")

# Функция для проверки влияния anomaly_wage с заданным порогом
def test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base):
    # Расчет аномалий
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly = thresh * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly).astype(int)

    # Создаем X с новым признаком
    X_train_temp = X_train_base.copy()
    X_test_temp = X_test_base.copy()
    X_train_temp['anomaly_wage'] = df_feat.loc[X_train_temp.index, 'anomaly_wage']
    X_test_temp['anomaly_wage'] = df_feat.loc[X_test_temp.index, 'anomaly_wage']

    # Масштабируем (вместе с новым признаком)
    scaler_temp = StandardScaler()
    X_train_scaled_temp = scaler_temp.fit_transform(X_train_temp)
    X_test_scaled_temp = scaler_temp.transform(X_test_temp)

    # Обучаем модель
    ridge_temp = Ridge(alpha=1.0)
    ridge_temp.fit(X_train_scaled_temp, y_train)
    y_pred_temp = ridge_temp.predict(X_test_scaled_temp)
    r2_temp = r2_score(y_test, y_pred_temp)

    return r2_temp, df_feat['anomaly_wage'].sum()

# Пробуем разные пороги
thresholds = [-1.0, -1.2, -1.5, -2.0]
best_r2_anomaly = r2_base
best_threshold = None
best_anomaly_count = 0

for thresh in thresholds:
    r2_temp, count = test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base)
    print(f"   Порог {thresh}σ: аномалий = {count}, R² = {r2_temp:.4f}")

    if r2_temp > best_r2_anomaly:
        best_r2_anomaly = r2_temp
        best_threshold = thresh
        best_anomaly_count = count

# Сохраняем лучшую версию anomaly_wage
if best_threshold is not None:
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly_best = best_threshold * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"\n   ✅ Лучший порог: {best_threshold}σ (аномалий: {best_anomaly_count})")
    print(f"   ✅ Улучшение R²: {best_r2_anomaly - r2_base:.4f}")

# 3.6. Лаги для WAGE, CPI, USDind
print("\n🔍 Добавление лагов для WAGE, CPI, USDind...")
for col in ['WAGE', 'CPI', 'USDind']:
    for lag in [1, 3, 6]:
        df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
print(f"   Добавлено 9 лаговых признаков (3 переменных × 3 лага)")

# 3.7. Взаимодействие UNEM и DEP1
print("\n🔍 Добавление взаимодействия UNEM × DEP1...")
df_feat['UNEM_DEP1'] = df_feat['UNEM'] * df_feat['DEP1']
print("   Добавлено взаимодействие UNEM × DEP1")

# ============================================================
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ
# ============================================================

print("\n" + "="*60)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ")
print("="*60)

# Удаляем вспомогательные столбцы
X_new = df_feat.drop(['DEPOS', 'DEPOS_log', 'Month', 'residual_wage'], axis=1).dropna()

print(f"📊 Число признаков в базовой модели: {X_base.shape[1]}")
print(f"📊 Число признаков в новой модели: {X_new.shape[1]}")
print(f"📊 Добавлено признаков: {X_new.shape[1] - X_base.shape[1]}")

# Разделение на train/test
X_train_new, X_test_new = X_new.iloc[:train_size], X_new.iloc[train_size:]

# Масштабирование
scaler_new = StandardScaler()
X_train_scaled_new = scaler_new.fit_transform(X_train_new)
X_test_scaled_new = scaler_new.transform(X_test_new)

# ============================================================
# 5. ОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ
# ============================================================

print("\n" + "="*60)
print("ОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ")
print("="*60)

ridge_new = Ridge(alpha=1.0)
ridge_new.fit(X_train_scaled_new, y_train)
y_pred_new = ridge_new.predict(X_test_scaled_new)

r2_new = r2_score(y_test, y_pred_new)
rmse_new = np.sqrt(mean_squared_error(y_test, y_pred_new))
mae_new = mean_absolute_error(y_test, y_pred_new)

print(f"\n📊 Результаты модели с новыми признаками:")
print(f"   R² = {r2_new:.4f}")
print(f"   RMSE = {rmse_new:.2f} млрд руб.")
print(f"   MAE = {mae_new:.2f} млрд руб.")

# ============================================================
# 6. СРАВНЕНИЕ МОДЕЛЕЙ
# ============================================================

print("\n" + "="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)

comparison = pd.DataFrame({
    'Модель': ['Базовая (Ridge)', 'С новыми признаками'],
    'R²': [r2_base, r2_new],
    'RMSE': [rmse_base, rmse_new],
    'MAE': [mae_base, mae_new],
    'Признаков': [X_base.shape[1], X_new.shape[1]]
})

print(comparison.to_string(index=False))

improvement = r2_new - r2_base
print(f"\n📊 Улучшение R²: {improvement:.4f}")

if improvement > 0.01:
    print("   ✅ Новые признаки ЗНАЧИТЕЛЬНО улучшают модель")
elif improvement > 0:
    print("   ✅ Новые признаки НЕМНОГО улучшают модель")
else:
    print("   ℹ️ Новые признаки НЕ улучшают модель")

# ============================================================
# 7. АНАЛИЗ КАЖДОГО НОВОГО ПРИЗНАКА
# ============================================================

print("\n" + "="*60)
print("АНАЛИЗ КАЖДОГО НОВОГО ПРИЗНАКА")
print("="*60)

def test_feature(X_train_base, X_test_base, feature_name, y_train, y_test, r2_base):
    X_train_test = X_train_base.copy()
    X_test_test = X_test_base.copy()
    X_train_test[feature_name] = X_train_new[feature_name]
    X_test_test[feature_name] = X_test_new[feature_name]

    scaler_test = StandardScaler()
    X_train_scaled_test = scaler_test.fit_transform(X_train_test)
    X_test_scaled_test = scaler_test.transform(X_test_test)

    ridge_test = Ridge(alpha=1.0)
    ridge_test.fit(X_train_scaled_test, y_train)
    y_pred_test = ridge_test.predict(X_test_scaled_test)
    r2_test = r2_score(y_test, y_pred_test)
    improvement = r2_test - r2_base
    return improvement

new_features = [
    'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12',
    'post_2022', 'covid', 'regime_cred1', 'anomaly_wage',
    'WAGE_lag_1', 'WAGE_lag_3', 'WAGE_lag_6',
    'CPI_lag_1', 'CPI_lag_3', 'CPI_lag_6',
    'USDind_lag_1', 'USDind_lag_3', 'USDind_lag_6',
    'UNEM_DEP1'
]

print("\n🔍 Влияние каждого нового признака на R²:")
results = []
for feature in new_features:
    imp = test_feature(X_train_base, X_test_base, feature, y_train, y_test, r2_base)
    results.append({'Признак': feature, 'Улучшение R²': imp})
    status = '✅' if imp > 0.001 else 'ℹ️'
    print(f"   {status} {feature}: {imp:.4f}")

results_df = pd.DataFrame(results).sort_values('Улучшение R²', ascending=False)
print("\n📊 Топ-5 признаков по улучшению:")
print(results_df.head(5).to_string(index=False))

# ============================================================
# 8. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("📌 ИТОГОВЫЙ ВЫВОД ПО БЛОКУ 4")
print("="*60)

best_r2 = max(r2_base, r2_new)
best_model = 'Базовая' if r2_base >= r2_new else 'С новыми признаками'

print(f"\n🏆 Лучшая модель: {best_model} (R² = {best_r2:.4f})")

if improvement > 0.01:
    print("\n✅ Добавление новых признаков ЗНАЧИТЕЛЬНО улучшает модель")
elif improvement > 0:
    print("\n✅ Добавление новых признаков НЕМНОГО улучшает модель")
else:
    print("\nℹ️ Добавление новых признаков НЕ улучшает модель")

print("\n📌 КЛЮЧЕВЫЕ ВЫВОДЫ:")
print("   1. Наибольшее улучшение дали признаки: " +
      ", ".join(results_df.head(3)['Признак'].tolist()))
print("   2. Сезонные переменные " +
      ("улучшают" if any('month' in r['Признак'] and r['Улучшение R²'] > 0.001 for r in results) else "не улучшают") + " модель")
print("   3. Фиктивная переменная post_2022 " +
      ("улучшает" if results_df[results_df['Признак'] == 'post_2022']['Улучшение R²'].values[0] > 0.001 else "не улучшает") + " модель")
print("   4. Лаги для WAGE, CPI, USDind " +
      ("улучшают" if any('_lag_' in r['Признак'] and r['Улучшение R²'] > 0.001 for r in results) else "не улучшают") + " модель")

# ============================================================
# 9. ВИЗУАЛИЗАЦИЯ ВЛИЯНИЯ ФИКТИВНЫХ ПЕРЕМЕННЫХ
# ============================================================

print("\n" + "="*60)
print("9. ВИЗУАЛИЗАЦИЯ ВЛИЯНИЯ ФИКТИВНЫХ ПЕРЕМЕННЫХ")
print("="*60)

# Получаем предсказания базовой модели и модели с новыми признаками
# (на всей выборке, чтобы видеть все точки)
X_full_base = X_base.copy()
X_full_new = X_new.copy()

# Масштабируем полные данные
scaler_full_base = StandardScaler()
X_full_scaled_base = scaler_full_base.fit_transform(X_full_base)

scaler_full_new = StandardScaler()
X_full_scaled_new = scaler_full_new.fit_transform(X_full_new)

# Предсказания
y_pred_full_base = ridge_base.predict(X_full_scaled_base)
y_pred_full_new = ridge_new.predict(X_full_scaled_new)

# Создаем DataFrame для визуализации
df_plot = df_feat.loc[X_full_new.index].copy()
df_plot['DEPOS_actual'] = y.loc[X_full_new.index]
df_plot['DEPOS_pred_base'] = y_pred_full_base
df_plot['DEPOS_pred_new'] = y_pred_full_new
df_plot['DEPOS_error_new'] = df_plot['DEPOS_actual'] - df_plot['DEPOS_pred_new']

# 9.1. График 1: DEPOS vs WAGE с адаптивным порогом аномалий
print("\n🔍 График 1: DEPOS vs WAGE (аномалии с адаптивным порогом)")

# Используем лучший порог из адаптивного подбора
if best_threshold is not None:
    threshold_anomaly_best = best_threshold * df_feat['residual_wage'].std()
    df_plot['anomaly_wage_best'] = (df_feat.loc[df_plot.index, 'residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"   Порог: {best_threshold}σ, аномалий: {df_plot['anomaly_wage_best'].sum()}")
else:
    threshold_anomaly_best = -1.5 * df_feat['residual_wage'].std()
    df_plot['anomaly_wage_best'] = (df_feat.loc[df_plot.index, 'residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"   Используем порог -1.5σ, аномалий: {df_plot['anomaly_wage_best'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Левый график: фактические данные с выделением аномалий
ax = axes[0]
scatter = ax.scatter(df_plot['WAGE'], df_plot['DEPOS_actual'],
                     c=df_plot['anomaly_wage_best'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('WAGE (руб.)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title(f'Фактические данные: аномалии (порог {best_threshold}σ)' if best_threshold else 'Фактические данные: аномалии')
ax.legend(*scatter.legend_elements(), title="anomaly")
ax.grid(True, alpha=0.3)

# Правый график: предсказанные значения с выделением аномалий
ax = axes[1]
scatter = ax.scatter(df_plot['WAGE'], df_plot['DEPOS_pred_new'],
                     c=df_plot['anomaly_wage_best'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('WAGE (руб.)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания модели: аномалии выделены')
ax.legend(*scatter.legend_elements(), title="anomaly")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_anomaly_wage.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.2. График 2: DEPOS vs UNEM с covid
print("\n🔍 График 2: DEPOS vs UNEM (сравнение моделей с covid и без)")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Левый график: предсказания без covid
ax = axes[0]
scatter = ax.scatter(df_plot['UNEM'], df_plot['DEPOS_pred_base'],
                     c=df_plot['covid'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('UNEM (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания БЕЗ covid')
ax.legend(*scatter.legend_elements(), title="covid")
ax.grid(True, alpha=0.3)

# Правый график: предсказания с covid
ax = axes[1]
scatter = ax.scatter(df_plot['UNEM'], df_plot['DEPOS_pred_new'],
                     c=df_plot['covid'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('UNEM (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания С covid')
ax.legend(*scatter.legend_elements(), title="covid")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_covid_unem.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.3. График 3: DEPOS vs CRED1 с регрессионными линиями для кластеров
print("\n🔍 График 3: DEPOS vs CRED1 (регрессионные линии для кластеров)")

# Разделение по DEPOS (правильное)
threshold_depos = 32000
mask1 = df_plot['DEPOS_actual'] <= threshold_depos
mask2 = df_plot['DEPOS_actual'] > threshold_depos

print(f"   Кластер 1 (DEPOS <= {threshold_depos}): {mask1.sum()} записей")
print(f"   Кластер 2 (DEPOS > {threshold_depos}): {mask2.sum()} записей")

# Функция для добавления регрессионной линии
def add_regression_line(ax, x, y, color, label):
    if len(x) < 2:
        print(f"   ⚠️ Недостаточно данных для регрессии: {label} (n={len(x)})")
        return
    model = LinearRegression()
    model.fit(x.values.reshape(-1, 1), y.values)
    x_range = np.linspace(x.min(), x.max(), 100)
    y_range = model.predict(x_range.reshape(-1, 1))
    ax.plot(x_range, y_range, color=color, linewidth=2, label=label)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Левый график: фактические данные
ax = axes[0]
ax.scatter(df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_actual'],
           color='blue', alpha=0.6, s=30, label='Кластер 1 (DEPOS ≤ 32000)')
add_regression_line(ax, df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_actual'],
                    'blue', 'Тренд кластера 1')
ax.scatter(df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_actual'],
           color='red', alpha=0.6, s=30, label='Кластер 2 (DEPOS > 32000)')
add_regression_line(ax, df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_actual'],
                    'red', 'Тренд кластера 2')
ax.axhline(y=threshold_depos, color='green', linestyle='--', alpha=0.5,
           label=f'Порог DEPOS = {threshold_depos}')
ax.set_xlabel('CRED1 (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Фактические данные: два кластера')
ax.legend()
ax.grid(True, alpha=0.3)

# Правый график: предсказания модели
ax = axes[1]
ax.scatter(df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_pred_new'],
           color='blue', alpha=0.6, s=30, label='Кластер 1')
add_regression_line(ax, df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_pred_new'],
                    'blue', 'Тренд предсказаний')
ax.scatter(df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_pred_new'],
           color='red', alpha=0.6, s=30, label='Кластер 2')
add_regression_line(ax, df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_pred_new'],
                    'red', 'Тренд предсказаний')
ax.axhline(y=threshold_depos, color='green', linestyle='--', alpha=0.5,
           label=f'Порог DEPOS = {threshold_depos}')
ax.set_xlabel('CRED1 (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания модели: два кластера')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_regime_cred1.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Визуализация завершена")


# ============================================================
# 10. ИСКЛЮЧЕНИЕ НЕЗНАЧИМЫХ ПРИЗНАКОВ (ИСПРАВЛЕННАЯ ВЕРСИЯ)
# ============================================================

print("\n" + "="*60)
print("10. ИСКЛЮЧЕНИЕ НЕЗНАЧИМЫХ ПРИЗНАКОВ")
print("="*60)

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from scipy.stats import shapiro

# -------------------------------------------------------------------
# 10.1. Ручное удаление признаков (на основе анализа)
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("ВАРИАНТ 1: Ручное удаление признаков")
print("-"*60)

# Список признаков для удаления (на основе анализа из key_insights_4)
features_to_drop = [
    'regime_cred1',      # ухудшает модель (-0.1148)
    'month_7',           # отрицательное влияние (-0.0027)
    'month_8',           # отрицательное влияние (-0.0104)
    'CPI_lag_6',         # отрицательное влияние (-0.0139)
    'USDind_lag_3',      # отрицательное влияние (-0.0185)
    'USDind_lag_1',      # отрицательное влияние (-0.0095)
    'WAGE_lag_6',        # отрицательное влияние (-0.0005)
    'CPI_lag_3',         # отрицательное влияние (-0.0032)
]

print(f"\n🔍 Удаляемые признаки ({len(features_to_drop)} шт.):")
for f in features_to_drop:
    print(f"   - {f}")

# Создаем X с исключенными признаками
X_manual = X_new.drop(columns=features_to_drop, axis=1)

# Разделение на train/test
X_train_manual, X_test_manual = X_manual.iloc[:train_size], X_manual.iloc[train_size:]

# Масштабирование
scaler_manual = StandardScaler()
X_train_scaled_manual = scaler_manual.fit_transform(X_train_manual)
X_test_scaled_manual = scaler_manual.transform(X_test_manual)

# Обучение модели
ridge_manual = Ridge(alpha=1.0)
ridge_manual.fit(X_train_scaled_manual, y_train)

# Предсказания на ОБЕИХ выборках
y_train_pred_manual = ridge_manual.predict(X_train_scaled_manual)
y_test_pred_manual = ridge_manual.predict(X_test_scaled_manual)

# Метрики на ТЕСТОВОЙ выборке (прогнозная способность)
r2_test_manual = r2_score(y_test, y_test_pred_manual)
rmse_test_manual = np.sqrt(mean_squared_error(y_test, y_test_pred_manual))
mae_test_manual = mean_absolute_error(y_test, y_test_pred_manual)

# Метрики на ОБУЧАЮЩЕЙ выборке (качество подгонки)
r2_train_manual = r2_score(y_train, y_train_pred_manual)

print(f"\n📊 Результаты (ручное удаление):")
print(f"   ОБУЧАЮЩАЯ выборка:")
print(f"     R²_train = {r2_train_manual:.4f}")
print(f"   ТЕСТОВАЯ выборка:")
print(f"     R²_test = {r2_test_manual:.4f}")
print(f"     RMSE = {rmse_test_manual:.2f} млрд руб.")
print(f"     MAE = {mae_test_manual:.2f} млрд руб.")
print(f"   Признаков: {X_manual.shape[1]}")

# -------------------------------------------------------------------
# 10.2. Stepwise Selection (пошаговый отбор)
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("ВАРИАНТ 2: Пошаговый отбор (Stepwise Selection)")
print("-"*60)

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

print("\n🔍 Выполняется пошаговый отбор (может занять несколько минут)...")

# Stepwise с использованием LinearRegression
sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select='auto',
    direction='forward',
    scoring='r2',
    cv=5,
    n_jobs=-1,
    tol=0.001
)

# Используем X_train_base (немасштабированные данные с именами)
sfs.fit(X_train_base, y_train)

# Получаем отобранные признаки
selected_features_stepwise = X_train_base.columns[sfs.get_support()].tolist()

print(f"\n📊 Отобрано {len(selected_features_stepwise)} признаков:")
print(f"   {selected_features_stepwise}")

# Создаем X с отобранными признаками
X_stepwise = X_new[selected_features_stepwise]

# Разделение на train/test
X_train_step, X_test_step = X_stepwise.iloc[:train_size], X_stepwise.iloc[train_size:]

# Масштабирование
scaler_step = StandardScaler()
X_train_scaled_step = scaler_step.fit_transform(X_train_step)
X_test_scaled_step = scaler_step.transform(X_test_step)

# Обучение модели
ridge_step = Ridge(alpha=1.0)
ridge_step.fit(X_train_scaled_step, y_train)

# Предсказания на ОБЕИХ выборках
y_train_pred_step = ridge_step.predict(X_train_scaled_step)
y_test_pred_step = ridge_step.predict(X_test_scaled_step)

# Метрики на ТЕСТОВОЙ выборке
r2_test_step = r2_score(y_test, y_test_pred_step)
rmse_test_step = np.sqrt(mean_squared_error(y_test, y_test_pred_step))
mae_test_step = mean_absolute_error(y_test, y_test_pred_step)

# Метрики на ОБУЧАЮЩЕЙ выборке
r2_train_step = r2_score(y_train, y_train_pred_step)

print(f"\n📊 Результаты (пошаговый отбор):")
print(f"   ОБУЧАЮЩАЯ выборка:")
print(f"     R²_train = {r2_train_step:.4f}")
print(f"   ТЕСТОВАЯ выборка:")
print(f"     R²_test = {r2_test_step:.4f}")
print(f"     RMSE = {rmse_test_step:.2f} млрд руб.")
print(f"     MAE = {mae_test_step:.2f} млрд руб.")
print(f"   Признаков: {X_stepwise.shape[1]}")

# Добавляем предсказания для полной Ridge модели на обучающей выборке
y_train_pred_new = ridge_new.predict(X_train_scaled_new)
r2_train_new = r2_score(y_train, y_train_pred_new)

print(f"\n📊 Результаты полной Ridge модели (для справки):")
print(f"   ОБУЧАЮЩАЯ выборка:")
print(f"     R²_train = {r2_train_new:.4f}")
print(f"   ТЕСТОВАЯ выборка:")
print(f"     R²_test = {r2_new:.4f}")
print(f"   Признаков: {X_new.shape[1]}")

# -------------------------------------------------------------------
# 10.3. РАСШИРЕННОЕ СРАВНЕНИЕ МОДЕЛЕЙ
# -------------------------------------------------------------------

print("\n" + "="*60)
print("РАСШИРЕННОЕ СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)

def calculate_aic_bic_corrected(y_true, y_pred, n_params):
    """
    Расчет AIC и BIC на обучающей выборке.

    Используем unbiased estimator для дисперсии: sigma2 = RSS / (n - n_params)
    Это дает более корректную оценку log-likelihood.

    Параметры:
    ----------
    y_true : array-like
        Фактические значения (обучающая выборка)
    y_pred : array-like
        Предсказанные значения (на обучающей выборке)
    n_params : int
        Количество параметров модели (включая intercept для Ridge)

    Возвращает:
    ----------
    aic, bic : float
        Значения информационных критериев
    """
    n = len(y_true)
    residuals = y_true - y_pred
    rss = np.sum(residuals**2)

    # ИСПРАВЛЕНИЕ: используем unbiased estimator для дисперсии
    sigma2 = rss / (n - n_params)  # unbiased estimator

    # Log-likelihood для нормального распределения
    log_likelihood = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1)

    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n)

    return aic, bic

def adjusted_r2_train(r2_train, n_train, n_params):
    """
    Расчет скорректированного R² на обучающей выборке.

    Параметры:
    ----------
    r2_train : float
        R² на обучающей выборке
    n_train : int
        Количество наблюдений в обучающей выборке
    n_params : int
        Количество параметров модели

    Возвращает:
    ----------
    adj_r2 : float
        Скорректированный R²
    """
    if n_train - n_params - 1 <= 0:
        return np.nan
    return 1 - (1 - r2_train) * (n_train - 1) / (n_train - n_params - 1)

n_train = len(y_train)  # 122
n_test = len(y_test)    # 12

# Для Ridge добавляем intercept как параметр
# Ridge включает intercept, если fit_intercept=True (по умолчанию)
intercept_param = 1

# Собираем все модели с данными для сравнения
models_data = [
    {
        'name': 'Полная Ridge (все признаки)',
        'y_train_pred': y_train_pred_new,
        'y_test_pred': y_pred_new,
        'n_params': X_new.shape[1] + intercept_param,
        'r2_train': r2_train_new,
        'r2_test': r2_new
    },
    {
        'name': 'Ручное удаление',
        'y_train_pred': y_train_pred_manual,
        'y_test_pred': y_test_pred_manual,
        'n_params': X_manual.shape[1] + intercept_param,
        'r2_train': r2_train_manual,
        'r2_test': r2_test_manual
    },
    {
        'name': 'Пошаговый отбор',
        'y_train_pred': y_train_pred_step,
        'y_test_pred': y_test_pred_step,
        'n_params': X_stepwise.shape[1] + intercept_param,
        'r2_train': r2_train_step,
        'r2_test': r2_test_step
    }
]

results_all = []

for model in models_data:
    # Метрики на обучающей выборке
    r2_train = model['r2_train']
    adj_r2 = adjusted_r2_train(r2_train, n_train, model['n_params'])
    aic, bic = calculate_aic_bic_corrected(y_train, model['y_train_pred'], model['n_params'])

    # Метрики на тестовой выборке (прогнозная способность)
    r2_test = model['r2_test']
    rmse_test = np.sqrt(mean_squared_error(y_test, model['y_test_pred']))
    mae_test = mean_absolute_error(y_test, model['y_test_pred'])

    # R²_gap: разница между качеством на обучающей и тестовой выборках
    r2_gap = r2_train - r2_test

    results_all.append({
        'Модель': model['name'],
        'Признаков': model['n_params'] - intercept_param,  # без intercept
        'R²_train': r2_train,
        'R²_adj_train': adj_r2,
        'R²_test': r2_test,
        'RMSE_test': rmse_test,
        'MAE_test': mae_test,
        'AIC': aic,
        'BIC': bic
    })

df_comparison = pd.DataFrame(results_all)

print("\n📊 Сравнительная таблица моделей (метрики на ОБУЧАЮЩЕЙ выборке):")
print("─" * 80)
print(f"{'Модель':<30} {'Призн.':<8} {'R²_train':<10} {'R²_adj':<10} {'AIC':<12} {'BIC':<12}")
print("─" * 80)
for _, row in df_comparison.iterrows():
    print(f"{row['Модель']:<30} {row['Признаков']:<8} {row['R²_train']:<10.4f} {row['R²_adj_train']:<10.4f} {row['AIC']:<12.2f} {row['BIC']:<12.2f}")

print("\n📊 Метрики на ТЕСТОВОЙ выборке (прогнозная способность):")
print("─" * 80)
print(f"{'Модель':<30} {'R²_test':<10} {'RMSE_test':<12} {'MAE_test':<12} {'R²_gap':<10}")
print("─" * 80)
for _, row in df_comparison.iterrows():
    print(f"{row['Модель']:<30} {row['R²_test']:<10.4f} {row['RMSE_test']:<12.2f} {row['MAE_test']:<12.2f} {row['R²_gap']:<10.4f}")

print("\n📌 R²_gap = R²_train - R²_test — разница между качеством на обучающей")
print("   и тестовой выборках. Большой R²_gap указывает на переобучение.")

# -------------------------------------------------------------------
# 10.4. ДИАГНОСТИКА ОСТАТКОВ НА ОБУЧАЮЩЕЙ ВЫБОРКЕ
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("ДИАГНОСТИКА ОСТАТКОВ (на ОБУЧАЮЩЕЙ выборке)")
print("-"*60)

def diagnose_residuals_train(y_train, y_train_pred, model_name):
    """
    Комплексная диагностика остатков на ОБУЧАЮЩЕЙ выборке.

    Проводимые тесты:
    1. Shapiro-Wilk test - нормальность остатков
    2. Breusch-Pagan test - гомоскедастичность
    3. Breusch-Godfrey test - автокорреляция (вместо DW)

    ВАЖНО: Все тесты проводятся на обучающей выборке!
    """
    residuals = y_train - y_train_pred
    n = len(residuals)

    # 1. Тест Шапиро-Уилка (нормальность)
    if n >= 3 and n <= 5000:  # Shapiro-Wilk работает для 3 <= n <= 5000
        shapiro_stat, shapiro_p = shapiro(residuals)
        normality = '✅ Нормально' if shapiro_p > 0.05 else '⚠️ Отклонение'
    else:
        shapiro_stat, shapiro_p = np.nan, np.nan
        normality = 'N/A (n вне диапазона)'

    # 2. Тест Бройша-Пагана (гомоскедастичность)
    try:
        # Используем предсказанные значения как экзогенную переменную
        exog_bp = sm.add_constant(y_train_pred.reshape(-1, 1))
        bp_stat, bp_p, bp_f, bp_f_p = het_breuschpagan(residuals, exog_bp)
        homoscedasticity = '✅ Гомоскедастично' if bp_p > 0.05 else '⚠️ Гетероскедастичность'
    except:
        bp_stat, bp_p = np.nan, np.nan
        homoscedasticity = 'Ошибка расчета'

    # 3. Тест Бройша-Годфри (автокорреляция) - ЗАМЕНА DW теста
    # Преимущества перед DW:
    # - Работает при наличии лаговых зависимых переменных
    # - Позволяет тестировать автокорреляцию более высоких порядков
    # - Более мощный для малых выборок
    try:
        # Создаем матрицу признаков (используем константу и предсказанные значения)
        exog_bg = sm.add_constant(y_train_pred.reshape(-1, 1))

        # Тест на автокорреляцию 1-го порядка (nlags=1)
        bg_stat, bg_p, bg_f, bg_f_p = acorr_breusch_godfrey(
            sm.OLS(residuals, exog_bg).fit(),
            nlags=1
        )
        autocorrelation = '✅ Нет автокорреляции' if bg_p > 0.05 else '⚠️ Автокорреляция'

        # Дополнительно тест на автокорреляцию 4-го порядка (сезонная)
        bg_stat_4, bg_p_4, _, _ = acorr_breusch_godfrey(
            sm.OLS(residuals, exog_bg).fit(),
            nlags=4
        )
        autocorrelation_4 = '✅ Нет' if bg_p_4 > 0.05 else '⚠️ Есть (порядок 4)'
    except:
        bg_stat, bg_p = np.nan, np.nan
        bg_stat_4, bg_p_4 = np.nan, np.nan
        autocorrelation = 'Ошибка расчета'
        autocorrelation_4 = 'Ошибка расчета'

    return {
        'Модель': model_name,
        'Shapiro-Wilk stat': shapiro_stat,
        'Shapiro-Wilk p': shapiro_p,
        'Нормальность': normality,
        'Breusch-Pagan p': bp_p,
        'Гомоскедастичность': homoscedasticity,
        'BG test (lag 1) p': bg_p,
        'Автокорреляция (lag 1)': autocorrelation,
        'BG test (lag 4) p': bg_p_4,
        'Автокорреляция (lag 4)': autocorrelation_4
    }

# Проводим диагностику для всех моделей
print("\n🔍 Проводится диагностика остатков...")
print("   ВАЖНО: Все тесты проводятся на ОБУЧАЮЩЕЙ выборке (n=122)")

diagnostic_results = []
for model in models_data:
    result = diagnose_residuals_train(
        y_train,
        model['y_train_pred'],
        model['name']
    )
    diagnostic_results.append(result)

df_diagnostic = pd.DataFrame(diagnostic_results)

print("\n📊 Результаты диагностики остатков (обучающая выборка):")
print("=" * 80)
for _, row in df_diagnostic.iterrows():
    print(f"\n{row['Модель']}:")
    print(f"  Нормальность: {row['Нормальность']} (p = {row['Shapiro-Wilk p']:.4f})")
    print(f"  Гомоскедастичность: {row['Гомоскедастичность']} (p = {row['Breusch-Pagan p']:.4f})")
    print(f"  Автокорреляция (lag 1): {row['Автокорреляция (lag 1)']} (p = {row['BG test (lag 1) p']:.4f})")
    print(f"  Автокорреляция (lag 4): {row['Автокорреляция (lag 4)']} (p = {row['BG test (lag 4) p']:.4f})")

# -------------------------------------------------------------------
# 10.5. ГРАФИКИ ОСТАТКОВ ДЛЯ ЛУЧШЕЙ МОДЕЛИ (на обучающей выборке)
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("ГРАФИКИ ОСТАТКОВ ДЛЯ ЛУЧШЕЙ МОДЕЛИ (обучающая выборка)")
print("-"*60)

# Определяем лучшую модель по AIC
best_aic_model = df_comparison.loc[df_comparison['AIC'].idxmin()]

print(f"\n📈 Лучшая модель по AIC: {best_aic_model['Модель']}")
print(f"   AIC = {best_aic_model['AIC']:.2f}")
print(f"   Признаков: {best_aic_model['Признаков']}")
print(f"   R²_train = {best_aic_model['R²_train']:.4f}")
print(f"   R²_test = {best_aic_model['R²_test']:.4f}")

# Находим соответствующие предсказания на обучающей выборке
model_mapping = {
    'Полная Ridge (все признаки)': y_train_pred_new,
    'Ручное удаление': y_train_pred_manual,
    'Пошаговый отбор': y_train_pred_step
}
y_train_pred_best = model_mapping[best_aic_model['Модель']]

# Рассчитываем остатки на ОБУЧАЮЩЕЙ выборке
residuals_train_best = y_train.values - y_train_pred_best

# Создаем графики
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Диагностика остатков (обучающая выборка, n={n_train})\n{best_aic_model["Модель"]}',
             fontsize=14, fontweight='bold')

# 1. Гистограмма остатков с кривой нормального распределения
ax = axes[0, 0]
ax.hist(residuals_train_best, bins=20, edgecolor='black', alpha=0.7, density=True)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Ноль')
# Накладываем кривую нормального распределения
from scipy.stats import norm
x_range = np.linspace(residuals_train_best.min(), residuals_train_best.max(), 100)
ax.plot(x_range, norm.pdf(x_range, residuals_train_best.mean(), residuals_train_best.std()),
        'r-', linewidth=2, label='Норм. распределение')
ax.set_xlabel('Остатки (млрд руб.)')
ax.set_ylabel('Плотность')
ax.set_title('Распределение остатков')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Q-Q plot
ax = axes[0, 1]
from scipy import stats
stats.probplot(residuals_train_best, dist="norm", plot=ax)
ax.set_title('Q-Q plot (проверка нормальности)')
ax.grid(True, alpha=0.3)

# 3. Остатки vs предсказанные значения
ax = axes[1, 0]
ax.scatter(y_train_pred_best, residuals_train_best, alpha=0.6, edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
# Добавляем сглаженную линию тренда
from scipy.interpolate import make_interp_spline
try:
    sorted_idx = np.argsort(y_train_pred_best)
    x_sorted = y_train_pred_best[sorted_idx]
    y_sorted = residuals_train_best[sorted_idx]
    # Простая скользящая средняя для визуализации тренда
    window = 20
    y_smooth = np.convolve(y_sorted, np.ones(window)/window, mode='valid')
    x_smooth = x_sorted[window//2:window//2+len(y_smooth)]
    ax.plot(x_smooth, y_smooth, 'g-', linewidth=2, label=f'Скользящее среднее (окно={window})')
except:
    pass
ax.set_xlabel('Предсказанные значения (млрд руб.)')
ax.set_ylabel('Остатки (млрд руб.)')
ax.set_title('Остатки vs Предсказанные (проверка гомоскедастичности)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Автокорреляционная функция остатков (ACF)
ax = axes[1, 1]
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(residuals_train_best, lags=min(20, n_train//4), ax=ax,
         title='Автокорреляционная функция остатков (ACF)')
ax.set_xlabel('Лаг')
ax.set_ylabel('Автокорреляция')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'residuals_diagnostics_{best_aic_model["Модель"].replace(" ", "_")}.png',
            dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Графики сохранены: residuals_diagnostics_{best_aic_model['Модель'].replace(' ', '_')}.png")

# -------------------------------------------------------------------
# 10.6. ИТОГОВЫЙ ВЫВОД (исправленный)
# -------------------------------------------------------------------

print("\n" + "="*60)
print("📌 ИТОГОВЫЙ ВЫВОД ПО СРАВНЕНИЮ МОДЕЛЕЙ")
print("="*60)

print("\n📊 ИТОГОВАЯ СВОДНАЯ ТАБЛИЦА:")
print("=" * 90)
print(f"{'Модель':<30} {'Призн.':<8} {'R²_train':<10} {'R²_adj':<10} {'R²_test':<10} {'AIC':<12} {'BIC':<12}")
print("=" * 90)
for _, row in df_comparison.iterrows():
    print(f"{row['Модель']:<30} {row['Признаков']:<8} {row['R²_train']:<10.4f} {row['R²_adj_train']:<10.4f} {row['R²_test']:<10.4f} {row['AIC']:<12.2f} {row['BIC']:<12.2f}")

# Определяем лучшие модели по разным критериям
best_r2_train = df_comparison.loc[df_comparison['R²_train'].idxmax()]
best_r2_test = df_comparison.loc[df_comparison['R²_test'].idxmax()]
best_aic = df_comparison.loc[df_comparison['AIC'].idxmin()]
best_bic = df_comparison.loc[df_comparison['BIC'].idxmin()]
best_adj_r2 = df_comparison.loc[df_comparison['R²_adj_train'].idxmax()]

print("\n" + "=" * 60)
print("🏆 ЛУЧШИЕ МОДЕЛИ ПО РАЗНЫМ КРИТЕРИЯМ:")
print("=" * 60)
print(f"📊 По R²_train (качество подгонки): {best_r2_train['Модель']} ({best_r2_train['R²_train']:.4f})")
print(f"📊 По R²_adj_train (с учетом сложности): {best_adj_r2['Модель']} ({best_adj_r2['R²_adj_train']:.4f})")
print(f"📊 По R²_test (прогнозная способность): {best_r2_test['Модель']} ({best_r2_test['R²_test']:.4f})")
print(f"📊 По AIC (информационный критерий): {best_aic['Модель']} ({best_aic['AIC']:.2f})")
print(f"📊 По BIC (с усиленным штрафом): {best_bic['Модель']} ({best_bic['BIC']:.2f})")

print("\n" + "=" * 60)
print("📌 РЕКОМЕНДАЦИЯ ПО ВЫБОРУ МОДЕЛИ:")
print("=" * 60)

# Анализ консенсуса критериев
criteria_winners = [
    best_r2_train['Модель'],
    best_adj_r2['Модель'],
    best_r2_test['Модель'],
    best_aic['Модель'],
    best_bic['Модель']
]

from collections import Counter
winner_counts = Counter(criteria_winners)
most_common_winner = winner_counts.most_common(1)[0]

print(f"\n📊 Консенсус критериев:")
for model, count in winner_counts.items():
    print(f"   {model}: {count} критериев из 5")

if most_common_winner[1] >= 3:
    print(f"\n✅ Большинство критериев (≥3) выбирают модель: {most_common_winner[0]}")
    recommended_model = most_common_winner[0]
else:
    print(f"\n⚠️ Критерии расходятся во мнениях")
    # При расхождении отдаем предпочтение AIC (баланс качество/сложность)
    recommended_model = best_aic['Модель']
    print(f"   Рекомендуется {recommended_model} (по AIC - баланс качество/сложность)")

print(f"\n🎯 РЕКОМЕНДУЕМАЯ МОДЕЛЬ: {recommended_model}")

# Выводим детальную информацию о рекомендуемой модели
rec_model_data = df_comparison[df_comparison['Модель'] == recommended_model].iloc[0]
print(f"\n📋 Характеристики рекомендуемой модели:")
print(f"   • Количество признаков: {rec_model_data['Признаков']}")
print(f"   • R² на обучении: {rec_model_data['R²_train']:.4f}")
print(f"   • R² скорректированный: {rec_model_data['R²_adj_train']:.4f}")
print(f"   • R² на тесте: {rec_model_data['R²_test']:.4f}")
print(f"   • AIC: {rec_model_data['AIC']:.2f}")
print(f"   • BIC: {rec_model_data['BIC']:.2f}")
print(f"   • RMSE на тесте: {rec_model_data['RMSE_test']:.2f} млрд руб.")

# Сравнение с полной Ridge моделью
if recommended_model != 'Полная Ridge (все признаки)':
    full_model_data = df_comparison[df_comparison['Модель'] == 'Полная Ridge (все признаки)'].iloc[0]
    reduction = full_model_data['Признаков'] - rec_model_data['Признаков']
    aic_diff = full_model_data['AIC'] - rec_model_data['AIC']
    print(f"\n📊 По сравнению с полной моделью:")
    print(f"   • Уменьшение числа признаков: {reduction} ({reduction/full_model_data['Признаков']*100:.1f}%)")
    print(f"   • Улучшение AIC: {aic_diff:.2f}")
    if rec_model_data['R²_test'] > full_model_data['R²_test']:
        print(f"   • Улучшение прогнозной способности: +{rec_model_data['R²_test'] - full_model_data['R²_test']:.4f}")
    else:
        print(f"   • Изменение прогнозной способности: {rec_model_data['R²_test'] - full_model_data['R²_test']:.4f}")

print("\n" + "=" * 60)
print("📌 КЛЮЧЕВЫЕ ВЫВОДЫ:")
print("=" * 60)
print("""
1. Все метрики качества подгонки (R²_train, R²_adj, AIC, BIC)
   рассчитаны на ОБУЧАЮЩЕЙ выборке для корректного сравнения моделей.

2. Прогнозная способность (R²_test, RMSE, MAE) оценена
   на ТЕСТОВОЙ выборке (12 последних наблюдений).

3. Диагностика остатков проведена на ОБУЧАЮЩЕЙ выборке:
   - Нормальность: тест Шапиро-Уилка
   - Гомоскедастичность: тест Бройша-Пагана
   - Автокорреляция: тест Бройша-Годфри (вместо DW)

4. F-тест исключен из анализа, так как он некорректен
   для Ridge-регрессии (смещенные оценки).

5. Тест Бройша-Годфри предпочтительнее DW-теста, так как:
   - Корректно работает при наличии лаговых переменных
   - Позволяет тестировать автокорреляцию разных порядков
   - Более мощный для малых выборок

6. Для итогового выбора модели используется консенсус
   нескольких критериев с приоритетом AIC.
""")

print("\n✅ Исключение незначимых признаков завершено")

print("\n✅ Блок 4 завершен")